# rewrite_data_files MVP — End-to-End Working Prototype

This notebook goes **all the way**: it implements a working `rewrite_data_files` compaction
using only existing pyiceberg primitives, demonstrates it end-to-end, and proposes the
architecture for the actual PR.

**Structure:**
1. Setup → create a table with the small-files problem
2. The MVP planner → full working Python code
3. The MVP rewriter → read + write using existing infrastructure
4. The MVP commit → atomic file swap
5. Verification → prove it worked
6. Written proposal → architecture + technical summary

---
## Part 1: Setup — Create a table with the small-files problem

In [1]:
import pyarrow as pa
import os, shutil, random, uuid
from collections import defaultdict

from pyiceberg.catalog.sql import SqlCatalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, LongType
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import IdentityTransform
from pyiceberg.expressions import AlwaysTrue, EqualTo

WAREHOUSE = "/tmp/iceberg_mvp_compaction"
DB_PATH = "/tmp/iceberg_mvp_compaction.db"
for path in [WAREHOUSE, DB_PATH]:
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)

catalog = SqlCatalog("test", uri=f"sqlite:///{DB_PATH}", warehouse=f"file://{WAREHOUSE}")
try:
    catalog.create_namespace("db")
except:
    pass

schema = Schema(
    NestedField(1, "id", LongType()),
    NestedField(2, "name", StringType()),
    NestedField(3, "category", StringType()),
    NestedField(4, "value", LongType()),
)
spec = PartitionSpec(
    PartitionField(source_id=3, field_id=1000, transform=IdentityTransform(), name="category")
)

table = catalog.create_table("db.events", schema=schema, partition_spec=spec)

# Create 18 small files (6 per partition) — exceeds MIN_INPUT_FILES=5
categories = ["electronics", "clothing", "food"]
for i in range(18):
    table.append(pa.table({
        "id": list(range(i*100, (i+1)*100)),
        "name": [f"item_{i}_{j}" for j in range(100)],
        "category": [categories[i % 3]] * 100,
        "value": [random.randint(1, 10000) for _ in range(100)],
    }))

print(f"✅ Table created with {len(list(table.scan().plan_files()))} data files")
print(f"   {table.scan().to_arrow().num_rows} total rows across 3 partitions")
print(f"   Snapshots: {len(table.metadata.snapshots)}")

✅ Table created with 18 data files
   1800 total rows across 3 partitions
   Snapshots: 18


In [2]:
# Inspect the state BEFORE compaction
print("=" * 80)
print("BEFORE COMPACTION")
print("=" * 80)

before_tasks = list(table.scan().plan_files())
before_files_by_part = defaultdict(list)
for t in before_tasks:
    before_files_by_part[str(t.file.partition)].append(t)

for part, tasks in before_files_by_part.items():
    total_size = sum(t.file.file_size_in_bytes for t in tasks)
    print(f"\n  Partition {part}: {len(tasks)} files, {total_size:,} bytes")
    for t in tasks:
        print(f"    {os.path.basename(t.file.file_path):45s} "
              f"{t.file.file_size_in_bytes:>6,} bytes  rows={t.file.record_count}")

total_files_before = len(before_tasks)
total_size_before = sum(t.file.file_size_in_bytes for t in before_tasks)
print(f"\n  TOTAL: {total_files_before} files, {total_size_before:,} bytes")

BEFORE COMPACTION

  Partition Record[food]: 6 files, 14,647 bytes
    00000-0-9e17c3fd-c1b3-477f-abb7-795e9283c275.parquet  2,424 bytes  rows=100
    00000-0-5b924c82-89e7-4caf-aa11-e24410e8da0d.parquet  2,423 bytes  rows=100
    00000-0-0686fc72-9f98-4ad7-af53-5762f5b61cd4.parquet  2,438 bytes  rows=100
    00000-0-d8d2b24c-04c5-47de-a9a6-a8d8072bc669.parquet  2,438 bytes  rows=100
    00000-0-ea904892-95c9-4432-8d16-3dd49fa0257e.parquet  2,441 bytes  rows=100
    00000-0-4531651e-b520-4b4a-b511-cce2cede6f6a.parquet  2,483 bytes  rows=100

  Partition Record[clothing]: 6 files, 14,737 bytes
    00000-0-6d909687-2638-4d3e-ad63-358ca59225fe.parquet  2,445 bytes  rows=100
    00000-0-71eb87ce-d699-4420-badb-3804224657ab.parquet  2,446 bytes  rows=100
    00000-0-6aeecb7c-46dd-4421-85f2-b5e122c68e28.parquet  2,454 bytes  rows=100
    00000-0-7c6d4641-2258-49f7-b755-15e23a27aa8d.parquet  2,455 bytes  rows=100
    00000-0-14d4e6a6-82ad-4e47-8f2b-15f74589f747.parquet  2,457 bytes  rows=100


---
## Part 2: The MVP Planner

This is the **new code** — the planning logic that mirrors Java's `BinPackRewriteFilePlanner`.
It identifies which files need compaction and groups them for rewriting.

In [3]:
from pyiceberg.table import TableProperties
from pyiceberg.utils.bin_packing import ListPacker
from pyiceberg.utils.properties import property_as_int
from dataclasses import dataclass, field
from typing import Optional


# ──────────────────────────────────────────────────────────
# Constants — mirrored from Java SizeBasedFileRewritePlanner
# ──────────────────────────────────────────────────────────
MIN_FILE_SIZE_DEFAULT_RATIO = 0.75
MAX_FILE_SIZE_DEFAULT_RATIO = 1.80
MIN_INPUT_FILES_DEFAULT = 5
MAX_FILE_GROUP_SIZE_BYTES_DEFAULT = 100 * 1024 * 1024 * 1024  # 100 GB


@dataclass
class RewriteGroup:
    """A group of files to be rewritten together.
    Mirrors Java's RewriteFileGroup."""
    partition_key: Optional[str]
    tasks: list
    
    @property
    def file_count(self) -> int:
        return len(self.tasks)
    
    @property
    def total_size(self) -> int:
        return sum(t.file.file_size_in_bytes for t in self.tasks)
    
    @property
    def total_records(self) -> int:
        return sum(t.file.record_count for t in self.tasks)


@dataclass
class RewriteResult:
    """Result of a rewrite_data_files operation."""
    rewritten_data_files_count: int = 0
    added_data_files_count: int = 0
    rewritten_bytes_count: int = 0
    failed_group_count: int = 0


def plan_rewrite_groups(table, row_filter=AlwaysTrue()):
    """
    Plan file groups for compaction.
    
    Mirrors the Java call chain:
      BinPackRewriteFilePlanner.plan()
        → planFileGroups()
          → scan + planFiles()          [Step 1]
          → groupByPartition()          [Step 2]
          → filterFiles()               [Step 3]
          → ListPacker.pack()           [Step 4]
          → filterFileGroups()          [Step 5]
    
    Args:
        table: The Iceberg Table object
        row_filter: Expression to select which files to consider
    
    Returns:
        List of RewriteGroup objects ready for execution
    """
    # ── Read table properties (same as Java's sizeThresholds + defaultTargetFileSize) ──
    target_file_size = property_as_int(
        properties=table.properties,
        property_name=TableProperties.WRITE_TARGET_FILE_SIZE_BYTES,
        default=TableProperties.WRITE_TARGET_FILE_SIZE_BYTES_DEFAULT,
    )
    min_file_size = int(target_file_size * MIN_FILE_SIZE_DEFAULT_RATIO)
    max_file_size = int(target_file_size * MAX_FILE_SIZE_DEFAULT_RATIO)
    
    print(f"  Thresholds: target={target_file_size:,}  min={min_file_size:,}  max={max_file_size:,}")
    
    # ── STEP 1: Scan with filter (mirrors table.newScan().filter(f).planFiles()) ──
    all_tasks = list(table.scan(row_filter=row_filter).plan_files())
    print(f"  Step 1 — Scanned {len(all_tasks)} files matching filter")
    
    # ── STEP 2: Group by partition (mirrors groupByPartition()) ──
    current_spec_id = table.spec().spec_id
    files_by_partition = defaultdict(list)
    for task in all_tasks:
        if task.file.spec_id == current_spec_id:
            key = str(task.file.partition)
        else:
            key = None  # incompatible spec → unpartitioned group
        files_by_partition[key].append(task)
    
    print(f"  Step 2 — Grouped into {len(files_by_partition)} partitions")
    
    # ── Process each partition (mirrors planFileGroups per partition) ──
    rewrite_groups = []
    
    for partition_key, tasks in files_by_partition.items():
        # ── STEP 3: Filter files by size (mirrors filterFiles + outsideDesiredFileSizeRange) ──
        filtered = [
            t for t in tasks
            if t.file.file_size_in_bytes < min_file_size
            or t.file.file_size_in_bytes > max_file_size
        ]
        
        if not filtered:
            continue  # all files in this partition are optimal size
        
        # ── STEP 4: Bin-pack (mirrors BinPacking.ListPacker.pack()) ──
        packer = ListPacker(
            target_weight=MAX_FILE_GROUP_SIZE_BYTES_DEFAULT,
            lookback=1,
            largest_bin_first=False,
        )
        groups = packer.pack(filtered, weight_func=lambda t: t.file.file_size_in_bytes)
        
        # ── STEP 5: Filter groups (mirrors filterFileGroups) ──
        for group in groups:
            group_size = sum(t.file.file_size_in_bytes for t in group)
            n = len(group)
            
            enough_files = n > 1 and n >= MIN_INPUT_FILES_DEFAULT
            enough_content = n > 1 and group_size > target_file_size
            too_much = group_size > max_file_size
            
            if enough_files or enough_content or too_much:
                rewrite_groups.append(RewriteGroup(
                    partition_key=partition_key,
                    tasks=group,
                ))
    
    print(f"  Steps 3-5 — {len(rewrite_groups)} groups ready to rewrite")
    for rg in rewrite_groups:
        print(f"    {rg.partition_key}: {rg.file_count} files, {rg.total_size:,} bytes, {rg.total_records} rows")
    
    return rewrite_groups


print("Planning compaction...")
groups = plan_rewrite_groups(table)
print(f"\n✅ Planning complete: {len(groups)} groups to rewrite")

Planning compaction...
  Thresholds: target=536,870,912  min=402,653,184  max=966,367,641
  Step 1 — Scanned 18 files matching filter
  Step 2 — Grouped into 3 partitions
  Steps 3-5 — 3 groups ready to rewrite
    Record[food]: 6 files, 14,647 bytes, 600 rows
    Record[clothing]: 6 files, 14,737 bytes, 600 rows
    Record[electronics]: 6 files, 14,779 bytes, 600 rows

✅ Planning complete: 3 groups to rewrite


In [4]:
# Also test with a FILTER — only compact one partition
print("Planning compaction with filter (category='electronics' only)...")
filtered_groups = plan_rewrite_groups(table, row_filter=EqualTo("category", "electronics"))
print(f"\n✅ Only {len(filtered_groups)} group(s) — other partitions untouched")

Planning compaction with filter (category='electronics' only)...
  Thresholds: target=536,870,912  min=402,653,184  max=966,367,641
  Step 1 — Scanned 6 files matching filter
  Step 2 — Grouped into 1 partitions
  Steps 3-5 — 1 groups ready to rewrite
    Record[electronics]: 6 files, 14,779 bytes, 600 rows

✅ Only 1 group(s) — other partitions untouched


---
## Part 3: The MVP Rewriter + Committer

Now we **execute** the plan: read each file group into Arrow, write back as compacted
files, and atomically swap the old files for the new ones.

In [5]:
import itertools
from pyiceberg.io.pyarrow import (
    _dataframe_to_data_files,
    ArrowScan,
)
from pyiceberg.table import Transaction


def rewrite_data_files(table, row_filter=AlwaysTrue()):
    """
    Full MVP implementation of rewrite_data_files.
    
    This mirrors the complete Java flow:
      1. Plan   → BinPackRewriteFilePlanner.plan()
      2. Execute → SparkBinPackFileRewriteRunner.doRewrite()
      3. Commit  → RewriteDataFilesCommitManager.commitFileGroups()
    
    Args:
        table: The Iceberg Table
        row_filter: Expression to filter which partitions/files to compact
    
    Returns:
        RewriteResult with statistics about what was done
    """
    result = RewriteResult()
    
    # ════════════════════════════════════════════════════════════════
    # PHASE 1: PLAN
    # ════════════════════════════════════════════════════════════════
    print("Phase 1: Planning...")
    rewrite_groups = plan_rewrite_groups(table, row_filter=row_filter)
    
    if not rewrite_groups:
        print("  No files need compaction.")
        return result
    
    # ════════════════════════════════════════════════════════════════
    # PHASE 2: EXECUTE — for each group: read files → write back
    # Mirrors SparkBinPackFileRewriteRunner.doRewrite()
    # ════════════════════════════════════════════════════════════════
    print("\nPhase 2: Executing rewrites...")
    
    all_old_files = []  # files to delete
    all_new_files = []  # files to add
    write_uuid = uuid.uuid4()
    counter = itertools.count(0)
    
    for i, group in enumerate(rewrite_groups):
        print(f"\n  Group {i+1}/{len(rewrite_groups)}: {group.partition_key}")
        print(f"    Reading {group.file_count} files ({group.total_size:,} bytes)...")
        
        # ── READ: Load all files in this group into an Arrow table ──
        # Java: spark.read().format("iceberg").load(groupId)
        # Python: ArrowScan reads specific FileScanTasks
        arrow_scan = ArrowScan(
            table_metadata=table.metadata,
            io=table.io,
            projected_schema=table.schema(),
            row_filter=AlwaysTrue(),
            case_sensitive=True,
            limit=None,
        )
        arrow_table = arrow_scan.to_table(group.tasks)
        print(f"    Read {arrow_table.num_rows} rows, {arrow_table.nbytes:,} bytes in memory")
        
        # ── WRITE: Write back using existing _dataframe_to_data_files ──
        # Java: spark.write().format("iceberg").save(groupId)
        # Python: _dataframe_to_data_files handles partitioning + bin-packing + Parquet writing
        new_files = list(_dataframe_to_data_files(
            table_metadata=table.metadata,
            df=arrow_table,
            io=table.io,
            write_uuid=write_uuid,
            counter=counter,
        ))
        
        old_files = [t.file for t in group.tasks]
        
        print(f"    Wrote {len(new_files)} new file(s):")
        for nf in new_files:
            print(f"      {os.path.basename(nf.file_path)}  {nf.file_size_in_bytes:,} bytes  "
                  f"rows={nf.record_count}  partition={nf.partition}")
        
        all_old_files.extend(old_files)
        all_new_files.extend(new_files)
        result.rewritten_data_files_count += len(old_files)
        result.added_data_files_count += len(new_files)
        result.rewritten_bytes_count += group.total_size
    
    # ════════════════════════════════════════════════════════════════
    # PHASE 3: COMMIT — atomic file swap
    # Mirrors RewriteDataFilesCommitManager.commitFileGroups()
    # Java: table.newRewrite().deleteFile(old).addFile(new).commit()
    # Python: _OverwriteFiles via transaction
    # ════════════════════════════════════════════════════════════════
    print(f"\nPhase 3: Committing ({len(all_old_files)} deletes + {len(all_new_files)} adds)...")
    
    with table.transaction() as tx:
        # Get the overwrite snapshot producer
        overwrite = tx.update_snapshot().overwrite()
        
        # Mark old files for deletion
        for old_file in all_old_files:
            overwrite.delete_data_file(old_file)
        
        # Add new compacted files
        for new_file in all_new_files:
            overwrite.append_data_file(new_file)
        
        overwrite.commit()
    
    print("  ✅ Committed!")
    
    # Reload table metadata to see the new snapshot
    table.refresh()
    
    return result

In [6]:
# ═══════════════════════════════════════════════════════════
# RUN IT!
# ═══════════════════════════════════════════════════════════
print("🚀 Running rewrite_data_files()...")
print("=" * 80)

result = rewrite_data_files(table)

print("\n" + "=" * 80)
print(f"📊 Result:")
print(f"   Rewritten: {result.rewritten_data_files_count} files ({result.rewritten_bytes_count:,} bytes)")
print(f"   Added:     {result.added_data_files_count} files")
print(f"   Net:       {result.rewritten_data_files_count} → {result.added_data_files_count} files")

🚀 Running rewrite_data_files()...
Phase 1: Planning...
  Thresholds: target=536,870,912  min=402,653,184  max=966,367,641
  Step 1 — Scanned 18 files matching filter
  Step 2 — Grouped into 3 partitions
  Steps 3-5 — 3 groups ready to rewrite
    Record[food]: 6 files, 14,647 bytes, 600 rows
    Record[clothing]: 6 files, 14,737 bytes, 600 rows
    Record[electronics]: 6 files, 14,779 bytes, 600 rows

Phase 2: Executing rewrites...

  Group 1/3: Record[food]
    Reading 6 files (14,647 bytes)...
    Read 600 rows, 22,596 bytes in memory
    Wrote 1 new file(s):
      00000-0-246b6898-b3ff-49b8-b153-a806361d9d86.parquet  6,213 bytes  rows=600  partition=Record[food]

  Group 2/3: Record[clothing]
    Reading 6 files (14,737 bytes)...
    Read 600 rows, 24,996 bytes in memory
    Wrote 1 new file(s):
      00000-1-246b6898-b3ff-49b8-b153-a806361d9d86.parquet  6,242 bytes  rows=600  partition=Record[clothing]

  Group 3/3: Record[electronics]
    Reading 6 files (14,779 bytes)...
    Read

---
## Part 4: Verification — Prove it worked

In [7]:
# Inspect the state AFTER compaction
print("=" * 80)
print("AFTER COMPACTION")
print("=" * 80)

after_tasks = list(table.scan().plan_files())
after_files_by_part = defaultdict(list)
for t in after_tasks:
    after_files_by_part[str(t.file.partition)].append(t)

for part, tasks in after_files_by_part.items():
    total_size = sum(t.file.file_size_in_bytes for t in tasks)
    print(f"\n  Partition {part}: {len(tasks)} files, {total_size:,} bytes")
    for t in tasks:
        print(f"    {os.path.basename(t.file.file_path):45s} "
              f"{t.file.file_size_in_bytes:>6,} bytes  rows={t.file.record_count}")

total_files_after = len(after_tasks)
total_size_after = sum(t.file.file_size_in_bytes for t in after_tasks)
print(f"\n  TOTAL: {total_files_after} files, {total_size_after:,} bytes")

AFTER COMPACTION

  Partition Record[food]: 1 files, 6,213 bytes
    00000-0-246b6898-b3ff-49b8-b153-a806361d9d86.parquet  6,213 bytes  rows=600

  Partition Record[clothing]: 1 files, 6,242 bytes
    00000-1-246b6898-b3ff-49b8-b153-a806361d9d86.parquet  6,242 bytes  rows=600

  Partition Record[electronics]: 1 files, 6,442 bytes
    00000-2-246b6898-b3ff-49b8-b153-a806361d9d86.parquet  6,442 bytes  rows=600

  TOTAL: 3 files, 18,897 bytes


In [8]:
# ═══════════════════════════════════════════════════════════
# VERIFICATION CHECKS
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("VERIFICATION")
print("=" * 80)

# 1. File count reduced
print(f"\n  ✓ File count: {total_files_before} → {total_files_after} "
      f"(reduced by {total_files_before - total_files_after})")
assert total_files_after < total_files_before, "File count should decrease!"

# 2. Data integrity — same rows, same values
df_before = table.scan().to_arrow().sort_by("id")
# We can't compare with before directly since we already committed,
# but we can verify row count and value integrity
print(f"  ✓ Row count preserved: {df_before.num_rows} rows")
assert df_before.num_rows == 1800, f"Expected 1800 rows, got {df_before.num_rows}"

# 3. Verify all IDs are present (0..1799)
ids = sorted(df_before.column("id").to_pylist())
assert ids == list(range(1800)), "Missing or duplicate IDs!"
print(f"  ✓ All IDs 0-1799 present and unique")

# 4. Verify partition integrity
for part, tasks in after_files_by_part.items():
    for t in tasks:
        assert str(t.file.partition) == part, f"File partition mismatch!"
print(f"  ✓ Partition integrity: all files in correct partitions")

# 5. New snapshot exists
latest_snapshot = table.metadata.snapshots[-1]
print(f"  ✓ New snapshot: {latest_snapshot.snapshot_id} (operation={latest_snapshot.summary})")

# 6. Categories still correct
for cat in categories:
    cat_rows = table.scan(row_filter=EqualTo("category", cat)).to_arrow().num_rows
    assert cat_rows == 600, f"Expected 600 rows for {cat}, got {cat_rows}"
print(f"  ✓ Per-partition row counts: electronics=600, clothing=600, food=600")

print("\n" + "=" * 80)
print("ALL CHECKS PASSED ✅")
print("=" * 80)


VERIFICATION

  ✓ File count: 18 → 3 (reduced by 15)
  ✓ Row count preserved: 1800 rows
  ✓ All IDs 0-1799 present and unique
  ✓ Partition integrity: all files in correct partitions
  ✓ New snapshot: 5325895459361064484 (operation=operation=Operation.OVERWRITE)
  ✓ Per-partition row counts: electronics=600, clothing=600, food=600

ALL CHECKS PASSED ✅


In [9]:
# ═══════════════════════════════════════════════════════════
# IDEMPOTENCY CHECK — running again should be a no-op
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("IDEMPOTENCY CHECK — running compaction again...")
print("=" * 80)

result2 = rewrite_data_files(table)

if result2.rewritten_data_files_count == 0:
    print("\n✅ IDEMPOTENT: No files needed compaction on second run!")
else:
    print(f"\n⚠️  Second run rewrote {result2.rewritten_data_files_count} files — "
          f"expected 0 for idempotency")
    print(f"   (This can happen if files are very small relative to the 512MB target — "
          f"they still fall below min_file_size)")


IDEMPOTENCY CHECK — running compaction again...
Phase 1: Planning...
  Thresholds: target=536,870,912  min=402,653,184  max=966,367,641
  Step 1 — Scanned 3 files matching filter
  Step 2 — Grouped into 3 partitions
  Steps 3-5 — 0 groups ready to rewrite
  No files need compaction.

✅ IDEMPOTENT: No files needed compaction on second run!


---
## Part 5: Test with filter — compact only one partition

In [10]:
# Reset: create fresh table
try:
    catalog.drop_table("db.events_filtered")
except:
    pass

table2 = catalog.create_table("db.events_filtered", schema=schema, partition_spec=spec)

for i in range(18):
    table2.append(pa.table({
        "id": list(range(i*100, (i+1)*100)),
        "name": [f"item_{i}_{j}" for j in range(100)],
        "category": [categories[i % 3]] * 100,
        "value": [random.randint(1, 10000) for _ in range(100)],
    }))

print(f"Fresh table: {len(list(table2.scan().plan_files()))} files")

# Compact ONLY electronics
print("\n🚀 Compacting only category='electronics'...")
result3 = rewrite_data_files(table2, row_filter=EqualTo("category", "electronics"))

after_tasks2 = list(table2.scan().plan_files())
after_by_part2 = defaultdict(list)
for t in after_tasks2:
    after_by_part2[str(t.file.partition)].append(t)

print(f"\nAfter selective compaction:")
for part in sorted(after_by_part2.keys()):
    tasks = after_by_part2[part]
    print(f"  {part}: {len(tasks)} files")

assert len(after_by_part2["Record[clothing]"]) == 6, "Clothing should be untouched!"
assert len(after_by_part2["Record[food]"]) == 6, "Food should be untouched!"
assert len(after_by_part2["Record[electronics]"]) < 6, "Electronics should be compacted!"
print("\n✅ Filter works: only electronics compacted, others untouched")

Fresh table: 18 files

🚀 Compacting only category='electronics'...
Phase 1: Planning...
  Thresholds: target=536,870,912  min=402,653,184  max=966,367,641
  Step 1 — Scanned 6 files matching filter
  Step 2 — Grouped into 1 partitions
  Steps 3-5 — 1 groups ready to rewrite
    Record[electronics]: 6 files, 14,773 bytes, 600 rows

Phase 2: Executing rewrites...

  Group 1/1: Record[electronics]
    Reading 6 files (14,773 bytes)...
    Read 600 rows, 26,696 bytes in memory
    Wrote 1 new file(s):
      00000-0-ee6d7e8a-c4f3-4548-b9b3-4b699bf572cd.parquet  6,431 bytes  rows=600  partition=Record[electronics]

Phase 3: Committing (6 deletes + 1 adds)...
  ✅ Committed!

After selective compaction:
  Record[clothing]: 6 files
  Record[electronics]: 1 files
  Record[food]: 6 files

✅ Filter works: only electronics compacted, others untouched


---
## Part 6: Written Proposal — Architecture & Technical Summary

### Issue #1092 — `rewrite_data_files` MVP Implementation Proposal

---

#### What this PR delivers

A bin-pack compaction API that fulfills the two requirements from the original issue:

1. **Predicate-based file selection**: Take a filter expression to find data files matching criteria
2. **Partition-grouped bin-pack rewriting**: Group files by partition and rewrite using the writer's existing constraints

---

#### Architecture

The implementation follows the same 3-phase architecture as the Java/Spark version:

```
┌──────────────────────────────────────────────────────────────────────┐
│                        User-Facing API                              │
│  table.maintenance.rewrite_data_files(filter=EqualTo("day", "...")) │
└──────────────────────────────────┬───────────────────────────────────┘
                                   │
                    ┌──────────────▼──────────────┐
                    │     Phase 1: PLAN            │
                    │  scan → group → filter →     │
                    │  bin-pack → filter groups     │
                    └──────────────┬──────────────┘
                                   │ List[RewriteGroup]
                    ┌──────────────▼──────────────┐
                    │     Phase 2: EXECUTE          │
                    │  ArrowScan.to_table() → read  │
                    │  _dataframe_to_data_files →   │
                    │  write                        │
                    └──────────────┬──────────────┘
                                   │ old_files, new_files
                    ┌──────────────▼──────────────┐
                    │     Phase 3: COMMIT          │
                    │  _OverwriteFiles:            │
                    │  delete old + add new →      │
                    │  atomic snapshot             │
                    └─────────────────────────────┘
```

---

#### Java ↔ Python Mapping

Every step mirrors the Java implementation:

| Step | Java Class & Method | Python Equivalent | Status |
|---|---|---|---|
| Scan | `table.newScan().filter().planFiles()` | `table.scan(row_filter=).plan_files()` | EXISTS |
| Group by partition | `BinPackRewriteFilePlanner.groupByPartition()` | `defaultdict` + loop with `spec_id` check | ~5 lines NEW |
| Filter files | `SizeBasedFileRewritePlanner.outsideDesiredFileSizeRange()` | `size < min or size > max` | ~3 lines NEW |
| Bin-pack | `BinPacking.ListPacker.pack()` | `ListPacker.pack()` | EXISTS |
| Filter groups | `BinPackRewriteFilePlanner.filterFileGroups()` | `enough_files \|\| enough_content \|\| too_much` | ~5 lines NEW |
| Read | `spark.read().format("iceberg")` | `ArrowScan.to_table(tasks)` | EXISTS |
| Write | `spark.write().format("iceberg")` | `_dataframe_to_data_files()` | EXISTS |
| Commit | `table.newRewrite().deleteFile().addFile().commit()` | `_OverwriteFiles.delete/append.commit()` | EXISTS |

---

#### Files Changed

| File | Change | Lines |
|---|---|---|
| `pyiceberg/table/maintenance.py` | Add `rewrite_data_files()` method | ~15 |
| `pyiceberg/table/compaction.py` (**NEW**) | Planner: `plan_rewrite_groups()`, `RewriteGroup`, `RewriteResult` | ~80 |
| `tests/table/test_compaction.py` (**NEW**) | Unit + integration tests | ~200 |

---

#### Configuration (from Java defaults)

| Parameter | Source | Default |
|---|---|---|
| `target_file_size` | `write.target-file-size-bytes` | 512 MB |
| `min_file_size` | 75% of target | 384 MB |
| `max_file_size` | 180% of target | 922 MB |
| `min_input_files` | Hardcoded (Java default) | 5 |
| `max_file_group_size` | Hardcoded (Java default) | 100 GB |

---

#### What's NOT in scope

These are explicitly deferred to follow-up PRs:

- ❌ Sort / Z-order rewrite strategies
- ❌ `partial-progress.enabled` (multi-commit)
- ❌ `delete-file-threshold` / `delete-ratio-threshold`
- ❌ `output-spec-id` override
- ❌ `max-concurrent-file-group-rewrites`
- ❌ `expectedOutputFiles()` remainder optimization
- ❌ Distributed execution (Ray/Dask)

Each of these layers naturally on top of the Phase 1 architecture.

---

#### Key Design Decisions

1. **Simple function API for v1**: `table.maintenance.rewrite_data_files(filter=...)` returns a
   `RewriteResult`. A fluent builder can be added later.

2. **Separate planner from executor**: `plan_rewrite_groups()` returns `RewriteGroup` objects
   that can be passed to any executor. This makes adding Ray/Dask straightforward in Phase 3.

3. **Reuse all existing primitives**: No new I/O code, no new Parquet handling, no new manifest
   logic. The only new code is the planning algorithm (~80 lines).

4. **Spec evolution handled correctly**: Files from old partition specs are grouped under a single
   "unpartitioned" key (matching Java's `groupByPartition()` logic) and re-partitioned under
   the current spec on write.

---

#### Verification Plan

| Test | What it verifies |
|---|---|
| `test_basic_compaction` | 18 small files → 3 compacted files, data intact |
| `test_filtered_compaction` | Only filtered partition compacted, others untouched |
| `test_idempotency` | Second run is a no-op (or produces minimal change) |
| `test_data_integrity` | All rows, columns, values preserved after compaction |
| `test_partition_integrity` | Files stay in correct partitions |
| `test_snapshot_created` | New snapshot with correct operation type |
| `test_empty_table` | No-op on empty table |
| `test_already_optimal` | No-op when all files are within size range |
| `test_unpartitioned_table` | Works on tables without partition spec |